In [ ]:
import torch 
import numpy as np

from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertModel, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import pandas as pd

def main():
    """
    Task B: Property Claim Modeling with Wisconsin LGPIF Dataset
    Implements a multi-task BERT model for:
    - B1: Hazard classification (9 classes: Vandalism, Fire, Lightning, Wind, Hail, Vehicle, WaterNW, WaterW, Misc)
    - B2: Log claim amount regression
    Dataset: peril.training.csv (~4,991 rows, one-hot encoded hazards, Loss, Description)
    For full dataset (~6,030 rows), download from:
    https://raw.githubusercontent.com/OpenActTexts/Loss-Data-Analytics/master/Data/PropertyFundInsample.csv
    """

    # Initialize tokenizer
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  # English for LGPIF

    # Load dataset
    try:
        df = pd.read_csv('peril.training.csv')  # Ensure file is in the same directory
    except FileNotFoundError:
        print("Error: 'peril.training.csv' not found. Please save the dataset in the same directory.")
        return

    # Extract hazard labels from one-hot columns
    hazard_columns = ['Vandalism', 'Fire', 'Lightning', 'Wind', 'Hail', 'Vehicle', 'WaterNW', 'WaterW', 'Misc']
    df['hazard_label'] = df[hazard_columns].idxmax(axis=1)
    label_encoder = LabelEncoder()
    df['hazard_label'] = label_encoder.fit_transform(df[hazard_columns].idxmax(axis=1))

    # Log-transform claim amount
    df['log_claim_amount'] = np.log1p(df['Loss'])

    # Prepare data
    X = df['Description']
    y_class = df['hazard_label']
    y_reg = df['log_claim_amount']

    # Split into train/test (80/20)
    X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
        X, y_class, y_reg, test_size=0.2, random_state=42
    )

    # Tokenize function
    def tokenize_texts(texts):
        return tokenizer(texts.tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')

    # Tokenize train and test sets
    train_encodings = tokenize_texts(X_train)
    test_encodings = tokenize_texts(X_test)

    # Create datasets
    train_dataset = TensorDataset(
        train_encodings['input_ids'],
        train_encodings['attention_mask'],
        torch.tensor(y_class_train.values, dtype=torch.long),
        torch.tensor(y_reg_train.values, dtype=torch.float)
    )
    test_dataset = TensorDataset(
        test_encodings['input_ids'],
        test_encodings['attention_mask'],
        torch.tensor(y_class_test.values, dtype=torch.long),
        torch.tensor(y_reg_test.values, dtype=torch.float)
    )

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=16)

    # Define multi-task model
    class BertMultiTask(nn.Module):
        def __init__(self, num_classes=9):
            super(BertMultiTask, self).__init__()
            self.bert = BertModel.from_pretrained('bert-base-uncased')
            self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)  # Hazard classification
            self.regressor = nn.Linear(self.bert.config.hidden_size, 1)  # Claim amount regression

        def forward(self, input_ids, attention_mask):
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            pooled_output = outputs.pooler_output
            class_logits = self.classifier(pooled_output)
            reg_output = self.regressor(pooled_output)
            return class_logits, reg_output

    # Initialize model
    model = BertMultiTask(num_classes=len(label_encoder.classes_))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # Optimizer and loss functions
    optimizer = AdamW(model.parameters(), lr=2e-5)
    class_loss_fn = nn.CrossEntropyLoss()
    reg_loss_fn = nn.MSELoss()

    # Training loop
    print("Training model...")
    for epoch in range(5):
        model.train()
        total_class_loss, total_reg_loss = 0, 0
        for batch in train_loader:
            input_ids, attention_mask, class_labels, reg_labels = [b.to(device) for b in batch]
            optimizer.zero_grad()
            class_logits, reg_output = model(input_ids, attention_mask)
            class_loss = class_loss_fn(class_logits, class_labels)
            reg_loss = reg_loss_fn(reg_output.squeeze(), reg_labels)
            loss = class_loss + reg_loss  # Equal weighting
            loss.backward()
            optimizer.step()
            total_class_loss += class_loss.item()
            total_reg_loss += reg_loss.item()
        print(f'Epoch {epoch+1}, Class Loss: {total_class_loss / len(train_loader):.4f}, Reg Loss: {total_reg_loss / len(train_loader):.4f}')

    # Evaluation
    print("\nEvaluating model...")
    model.eval()
    class_preds, reg_preds = [], []
    true_class, true_reg = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids, attention_mask, class_labels, reg_labels = [b.to(device) for b in batch]
            class_logits, reg_output = model(input_ids, attention_mask)
            class_preds.extend(torch.argmax(class_logits, dim=1).cpu().numpy())
            reg_preds.extend(reg_output.squeeze().cpu().numpy())
            true_class.extend(class_labels.cpu().numpy())
            true_reg.extend(reg_labels.cpu().numpy())

    # Compute metrics
    f1 = f1_score(true_class, class_preds, average='weighted')
    rmse = np.sqrt(mean_squared_error(true_reg, reg_preds))
    print(f'B1 Test F1-Score (Hazard): {f1:.4f}')
    print(f'B2 Test RMSE (Log Claim Amount): {rmse:.4f}')

    # Inference function
    def predict(description):
        model.eval()
        inputs = tokenizer(description, padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            class_logits, reg_output = model(inputs['input_ids'], inputs['attention_mask'])
        hazard = label_encoder.inverse_transform([torch.argmax(class_logits, dim=1).cpu().numpy()[0]])[0]
        claim_amount = np.expm1(reg_output.squeeze().cpu().numpy())  # Reverse log-transform
        return hazard, claim_amount

    # Example inference
    print("\nExample prediction:")
    description = "lightning damage to courthouse"
    hazard, claim_amount = predict(description)
    print(f'Predicted Hazard: {hazard}, Predicted Claim Amount: ${claim_amount:.2f}')

    # Notes for improvement
    print("\nNotes for improvement:")
    print("- Use full LGPIF dataset (~6,030 rows) for better results: https://raw.githubusercontent.com/OpenActTexts/Loss-Data-Analytics/master/Data/PropertyFundInsample.csv")
    print("- Handle class imbalance with weighted loss:")
    print("  class_counts = df[hazard_columns].sum()")
    print("  class_weights = 1 / class_counts")
    print("  class_weights = torch.tensor(class_weights.values, dtype=torch.float).to(device)")
    print("  class_loss_fn = nn.CrossEntropyLoss(weight=class_weights)")
    print("- Tune hyperparameters (learning rate, batch size, epochs) with Hugging Face Trainer.")
    print("- Add validation set (e.g., peril.validation.csv) for robust evaluation.")
    print("- Use distilbert-base-uncased or freeze BERT layers for faster training:")
    print("  for param in model.bert.encoder.layer[:6].parameters():")
    print("      param.requires_grad = False")
    print("- Save model for deployment: torch.save(model.state_dict(), 'model.pt')")

if __name__ == "__main__":
    main()
